In [32]:
import torch.nn as nn
import torchvision.models as models

#1. Load pretrained ResNet
resnet = models.resnet18(pretrained=True)
for param in resnet.parameters(): param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 10) # replace head

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# Data transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 training dataset
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_dl = DataLoader(train_data, batch_size=64, shuffle=True)

# Load CIFAR-10 test dataset (optional but good practice)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_dl = DataLoader(test_data, batch_size=64, shuffle=False)

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet.to(device)

#2. Train only final layer
opt = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)
for ep in range(2):
    for x,y in train_dl:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(); out = resnet(x); loss = F.cross_entropy(out,y)
        loss.backward(); opt.step()
    print(f"Epoch {ep+1} done")

Epoch 1 done


In [36]:
import torch

def accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [37]:
#3. Evaluate
print("Transfer learning test acc:", accuracy(resnet, test_dl))

Transfer learning test acc: 0.4566
